# Modelado 

In [1]:
import pandas as pd
import numpy as np 
import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn import metrics  

import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore') 

C:\Users\SANTIAGO\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def carga_info(data):
    drop_cols = [
        'Municipio', 
        'Departamento', 
        'EVI_Min_Date', 
        'EVI_Max_Date', 
        'NDVI_Min_Date', 
        'NDVI_Max_Date', 
        'NDVI_Max_Count', 
        #'Latitude', 
        #'Longitude', 
        'Producto', 
        'Área Cosechada', 
        'Área Sembrada', 
        'Ciclo',
        'Producción'
        ]
    
    #Creación de nuevas variables necesarias
    data['Mpio'] = data['Municipio'].str.strip()+'_'+data['Departamento'].str.strip()
    data['Flag_covid'] = np.where(data['Year']==2020, 1, 0)
    data.rename(columns={'EVI_Max_Count':'Resolucion'}, inplace=True)
    #Eliminación de columnas innecesarias
    data.drop(columns=drop_cols, inplace=True)
    #Eliminación de nuelos -- esto puede cambiar si decidimos hacer imputacion
    data.dropna(inplace=True)
    #Ordenamiento
    data = data[[
        'Year', 'Mpio', 
        't2m_mean', 'd2m_mean', 'tp_mean', 'ssrd_mean', 'e_mean', 'stl1_mean', 
        't2m_std', 'd2m_std', 'tp_std', 'ssrd_std', 'e_std', 'stl1_std', 
        'Resolucion',
       'EVI_Min_Minimum', 'EVI_Max_Maximum', 'EVI_Avg_StdDev', 'EVI_Avg_Mean_deviation', 'EVI_Avg_StdDev_deviation',
       'EVI_Median_Median', 'EVI_Avg_Mean', 'NDVI_Min_Minimum', 'NDVI_Avg_Mean_deviation', 'NDVI_Avg_StdDev_deviation',
       'NDVI_Max_Maximum', 'NDVI_Avg_StdDev', 'NDVI_Median_Median',
       'NDVI_Avg_Mean', 'Flag_covid', 'Rendimiento']]
    
    print(f'Data cargada para {data}')
    data.info()
    return data

def univariate_ols_analysis(df, target='Rendimiento', test_size=0.2, random_state=42, verbose=True):
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        df.drop(columns=[target]),
        df[target],
        test_size=test_size,
        random_state=random_state
    )
    
    predictors = X_train.columns
    results = []
    
    for pred in predictors:
        
        X = X_train[[pred]].copy()
        
        # Convertir categóricas a dummies
        if X.dtypes[0] == 'object':
            X = pd.get_dummies(X, drop_first=True)
        
            # Agregar intercepto
        X = sm.add_constant(X)
        
        # Fit modelo
        mod = sm.OLS(y_train, X)
        res = mod.fit()
        
        if verbose:
            print(f"\nResultados para predictor: {pred}")
            print(res.summary())
        
        # Extraer coeficiente y p-value (puede haber varias columnas si hubo dummies)
        for var in X.columns:
            if var != 'const':
                results.append({
                    'predictor': pred,
                    'variable_model': var,
                    'R2': res.rsquared,
                    'F_stat': res.fvalue,
                    'coef': res.params[var],
                    'p_value': res.pvalues[var],
                })
    
    df_results = pd.DataFrame(results)
    df_results.sort_values(by='R2', ascending=False, inplace=True)
    df_results.reset_index(drop=True, inplace=True)
    
    return df_results

def univariate_ols_analysis(df, target='Rendimiento', test_size=0.2, random_state=42, verbose=True):
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        df.drop(columns=[target]),
        df[target],
        test_size=test_size,
        random_state=random_state
    )
    
    results = []
    
    for pred in X_train.columns:
        
        X = X_train[[pred]].copy()
        
        # Convertir categóricas a dummies SI es necesario
        if X.dtypes.iloc[0] == 'object' or str(X.dtypes.iloc[0]).startswith('category'):
            X = pd.get_dummies(X, drop_first=True)
        
        # 🔑 FORZAR a numérico (CLAVE para evitar tu error)
        X = X.apply(pd.to_numeric, errors='coerce')
        y = pd.to_numeric(y_train, errors='coerce')
        
        # Eliminar NaNs generados
        data = pd.concat([X, y], axis=1).dropna()
        X_clean = data.drop(columns=[target])
        y_clean = data[target]
        
        # Agregar constante
        X_clean = sm.add_constant(X_clean)
        
        # Fit
        try:
            mod = sm.OLS(y_clean, X_clean)
            res = mod.fit()
        except Exception as e:
            print(f"Error con predictor {pred}: {e}")
            continue
        
        if verbose:
            print(f"\nResultados para predictor: {pred}")
            print(res.summary())
        
        # Extraer resultados
        for var in X_clean.columns:
            if var != 'const':
                results.append({
                    'predictor': pred,
                    'variable_model': var,
                    'R2': res.rsquared,
                    'F_stat': res.fvalue,
                    'coef': res.params[var],
                    'p_value': res.pvalues[var],
                })
    
    df_results = pd.DataFrame(results)
    df_results.sort_values(by='R2', ascending=False, inplace=True)
    df_results.reset_index(drop=True, inplace=True)
    
    return df_results

def plot_corr_with_target(df, target='Rendimiento', drop_cols=None, plot=True):
    
    # Copia del df
    data = df.copy()
    
    # Eliminar columnas si se especifica
    if drop_cols:
        data = data.drop(columns=drop_cols)
    
    # Mantener solo numéricas (evita errores)
    data = data.select_dtypes(include='number')
    
    # Correlación
    corr_df = data.corr()
    
    # Plot
    if plot:
        plt.figure(figsize=(16,9))
        sns.heatmap(
            corr_df,
            cmap='coolwarm',
            center=0,
            annot=True,
            fmt=".2f",
            linewidths=0.2
        )
        plt.title("Correlation Matrix")
        plt.show()
    
    # Ranking respecto al target
    corr_target = corr_df[target].sort_values(ascending=False)
    
    return corr_df, corr_target

def random_forest_feature_importance(
    df,
    target='Rendimiento',
    drop_cols=None,
    test_size=0.2,
    random_state=42,
    n_estimators=100
):
    
    # Copia del dataframe
    data = df.copy()
    
    # Eliminar columnas si aplica
    if drop_cols:
        data = data.drop(columns=drop_cols)
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        data.drop(columns=[target]),
        data[target],
        test_size=test_size,
        random_state=random_state
    )
    
    # Modelo
    RF = RandomForestRegressor(
        random_state=random_state,
        n_estimators=n_estimators
    )
    
    RF.fit(X_train, y_train)
    
    # Feature importance
    feat_imp_df = pd.DataFrame({
        'feature': RF.feature_names_in_,
        'importance': RF.feature_importances_
    })
    
    feat_imp_df = feat_imp_df.sort_values(by='importance', ascending=False).reset_index(drop=True)
    
    return feat_imp_df, RF

In [3]:
#Selección de variables
feats_select_regresion=[
 'EVI_Avg_Mean_deviation',
 'ssrd_mean',
 'EVI_Median_Median',
 'ssrd_std',
 'd2m_std',
 'EVI_Avg_Mean',
 'NDVI_Max_Maximum',
 'EVI_Max_Maximum',
 'e_std'
 ]

feats_select_imp = [
 'EVI_Avg_Mean_deviation',
 'ssrd_mean',
 'ssrd_std',
 'EVI_Median_Median',
 'd2m_std',
 'e_std',
 'NDVI_Avg_Mean_deviation',
 'stl1_std',
 'tp_std',
 'NDVI_Max_Maximum',
 'tp_mean']

In [4]:
df_ago_nov = pd.read_excel("../Data anual/data_anual_ago_nov.xlsx")
df_sep_dec = pd.read_excel("../Data anual/data_anual_sep_dec.xlsx")
df_total = pd.read_excel("../Data anual/data_anual_total.xlsx")

df_ago_nov = carga_info(df_ago_nov)
df_sep_dec = carga_info(df_sep_dec)
df_total = carga_info(df_total)

Data cargada para      Year                         Mpio  t2m_mean  d2m_mean   tp_mean  \
0    2007             MONIQUIRA_BOYACA   14.8025   11.7700  0.007746   
1    2008             MONIQUIRA_BOYACA   14.8300   11.9650  0.007302   
2    2009             MONIQUIRA_BOYACA   15.7175   11.6800  0.005707   
3    2010             MONIQUIRA_BOYACA   15.0000   12.2750  0.008864   
4    2011             MONIQUIRA_BOYACA   15.0975   11.8325  0.007635   
..    ...                          ...       ...       ...       ...   
427  2020  VALLE DE SAN JOSE_SANTANDER   18.2800   15.5525  0.015731   
428  2021  VALLE DE SAN JOSE_SANTANDER   18.3925   15.3675  0.013376   
429  2022  VALLE DE SAN JOSE_SANTANDER   18.0450   15.2600  0.013582   
430  2023  VALLE DE SAN JOSE_SANTANDER   19.1375   16.0350  0.012559   
431  2024  VALLE DE SAN JOSE_SANTANDER   18.9775   15.9050  0.014026   

      ssrd_mean    e_mean  stl1_mean   t2m_std   d2m_std  ...  EVI_Avg_Mean  \
0    17856799.0 -0.003116    14.9900  

In [22]:
df = df_total
feats = feats_select_regresion
target = 'Rendimiento'
model = LinearRegression()
run_name = 'Total months - Regression Selection'

In [26]:
    
def model_evaluation(df, target, feats, model, run_name):
    #Registros fuera del tiempo (OOT)
    df_oot = df[df['Year'].isin([2024])]
    df_oot = df_oot[[target,'Flag_covid','Mpio']+feats]
    df_oot_dummies = pd.get_dummies(df_oot, drop_first=True)
    X_oot = df_oot_dummies.drop(columns=[target])
    X_oot = X_oot.reindex(columns=X_oot.columns, fill_value=0)
    y_oot = df_oot_dummies[[target]]

    #Registros en el tiempo (IT)
    df_it = df[~df['Year'].isin([2024])]
    df_it = df_it[[target,'Flag_covid','Mpio']+feats]
    df_it_dummies = pd.get_dummies(df_it, drop_first=True)

    X = df_it_dummies.drop(columns=[target])
    Y= df_it_dummies[[target]]

    X_train, X_test, y_train, y_test = train_test_split(X, Y , test_size=0.2, random_state=27)

    #Entrenamiento del modelo
    model.fit(X_train,y_train)

    #Predicciones 
    y_pred_it_train = model.predict(X_train)
    y_pred_it_test = model.predict(X_test)
    y_pred_oot = model.predict(X_oot)

    #Evaluación del modelo
    mse_it_train = metrics.mean_squared_error(y_train.values.ravel(), y_pred_it_train)
    mse_it_test = metrics.mean_squared_error(y_test.values.ravel(), y_pred_it_test)
    mse_oot = metrics.mean_squared_error(y_oot.values.ravel(), y_pred_oot)

    rmse_it_train = np.sqrt(mse_it_train)
    rmse_it_test = np.sqrt(mse_it_test)
    rmse_oot = np.sqrt(mse_oot)

    df_results = pd.DataFrame({
        'run_name' : [run_name],
        'MSE_IT_Train': [mse_it_train],
        'MSE_IT_Test': [mse_it_test],
        'MSE_OOT': [mse_oot],
        'RMSE_IT_Train': [rmse_it_train],
        'RMSE_IT_Test': [rmse_it_test],
        'RMSE_OOT': [rmse_oot]
    })

    return df_results, model

In [28]:
df_results, model = model_evaluation(df = df_total,feats = feats_select_regresion, target = 'Rendimiento', model = LinearRegression(), run_name = 'Total months - Regression Selection')
df_results

,run_name,MSE_IT_Train,MSE_IT_Test,MSE_OOT,RMSE_IT_Train,RMSE_IT_Test,RMSE_OOT
0,Total months - Regression Selection,0.051073,0.08127,0.066975,0.225993,0.285079,0.258796
